# Offline d1/d2/d3 validation harness

Scores any retrieval method on synthetic proxies of all three difficulty levels,
built from dataset1's labelled pairs. **Run All** and read the table at the bottom.

- Lives in `amine/` alongside `eval_harness.py` (kernel cwd = this folder).
- Cell 2 symlinks `data/` and `out/` here so the data + outputs show in the left panel.
- Edit `EMBEDDERS` in `eval_harness.py` to plug in new methods, then re-run.
- Last cell gives download links for the outputs.

In [ ]:
import subprocess, sys
# scipy + nibabel are all the harness needs (no GPU for the reference embedders).
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nibabel>=5.3', 'scipy'], check=True)
print('deps ready')

In [ ]:
import os
# Make the input data and output dir browsable in the file panel on the left:
# create symlinks INSIDE this folder (amine/) pointing at the real locations.
# One-time per container; harmless if they already exist.
for name, target in [('data', '/workspace/data/ehl'), ('out', '/workspace/out')]:
    if os.path.islink(name) or os.path.exists(name):
        print('exists:', name, '->', os.path.realpath(name))
        continue
    if os.path.isdir(target):
        os.symlink(target, name)
        print('linked:', name, '->', target)
    else:
        print('SKIP', name, '- target not found:', target, '(fix the path if your mount differs)')

In [ ]:
import os
# The box mounts the data here (see CLAUDE_workflow.md). Adjust if your mount differs.
os.environ['DATA_ROOT'] = '/workspace/data/ehl'
os.environ.setdefault('N_VAL', '60')   # held-out dataset1 pairs to score on
os.environ.setdefault('GRID', '96')    # resample cube size
assert os.path.isdir(os.environ['DATA_ROOT']), 'data root not found - fix DATA_ROOT above'
# eval_harness.py is uploaded into the jupyter root, which is also the kernel cwd.
assert os.path.exists('eval_harness.py'), \
    'eval_harness.py not found in cwd - re-upload it (python tools/jupyter_put.py eval_harness.py)'
print('using', os.environ['DATA_ROOT'])

In [ ]:
import importlib, eval_harness
importlib.reload(eval_harness)   # pick up edits without restarting the kernel
results = eval_harness.evaluate()

In [ ]:
import os, shutil
from IPython.display import FileLink, display
# Click these links to download outputs to your computer.
# (submission.csv is produced by run_baseline.ipynb; searched in a few common spots.)
def offer(fname, candidates):
    for c in candidates:
        if os.path.exists(c):
            if os.path.abspath(c) != os.path.abspath(fname):
                shutil.copy(c, fname)   # bring it next to this notebook so the link resolves
            display(FileLink(fname))
            return
    print('not found yet:', fname)

offer('eval_results.md', ['eval_results.md'])
offer('submission.csv', ['submission.csv', 'out/submission.csv',
                         '/workspace/out/submission.csv', '/root/submission.csv'])

In [ ]:
from IPython.display import Markdown
Markdown(open('eval_results.md').read())